In [0]:
spark.catalog.setCurrentCatalog("futebol_analytics")

In [0]:
from pyspark.sql.functions import (
    col,
    current_timestamp,
    initcap,
    lower,
    round,
    to_date,
    trim,
)

# Define o catálogo ativo e assegura a existência do schema Silver
spark.catalog.setCurrentCatalog("futebol_analytics")
spark.sql("CREATE SCHEMA IF NOT EXISTS silver")
print(
    "[CONFIG] Catálogo configurado em 'futebol_analytics' e schema 'silver'"
    " validado."
)

In [0]:
print("Processando e consolidando avaliações da FotMob...")

# 1. Unificação das 4 safras anuais da Bronze
df_fotmob_raw = (
    spark.table("bronze.fotmob_rating_2022")
    .unionByName(spark.table("bronze.fotmob_rating_2023"))
    .unionByName(spark.table("bronze.fotmob_rating_2024"))
    .unionByName(spark.table("bronze.fotmob_rating_2025"))
)

# 2. Aplicação de regras de qualidade e padronização
df_fotmob_silver = (
    df_fotmob_raw
    # Unicidade: Elimina duplicatas de um mesmo atleta na mesma partida
    .dropDuplicates(["match_id", "player_id"])
    # Padronização textual para viabilizar JOINs semânticos
    .withColumn("player_name_clean", lower(trim(col("player_name"))))
    .withColumn("team_name_clean", trim(col("team_name")))
    # Acurácia e Completude: Notas válidas na escala de 0 a 10 e remoção de registros órfãos
    .filter(
        col("rating").isNotNull() & (col("rating") >= 0) & (col("rating") <= 10)
    )
    .filter(col("player_name").isNotNull())
    .withColumn("rating", round(col("rating"), 2))
    # Auditoria
    .withColumn("_transformed_at", current_timestamp())
    .select(
        col("match_id"),
        col("match_time_utc"),
        col("season"),
        col("match_year"),
        col("player_id").alias("fotmob_player_id"),
        col("player_name"),
        col("player_name_clean"),
        col("team_name_clean").alias("team_name"),
        col("is_goalkeeper"),
        col("position_id"),
        col("rating"),
        col("_transformed_at"),
    )
)

# 3. Persistência na camada Silver
df_fotmob_silver.write.format("delta").mode("overwrite").saveAsTable(
    "silver.fotmob_ratings"
)
print(
    f"[OK] Tabela 'silver.fotmob_ratings' criada com"
    f" {spark.table('silver.fotmob_ratings').count():,} registros."
)

In [0]:
print("Processando dados cadastrais de atletas (Kaggle: players)...")

df_players_silver = (
    spark.table("bronze.players")
    # Unicidade: Chave primária de jogador única
    .dropDuplicates(["player_id"])
    # Consistência: Conversão de datas gravadas como string para DateType real
    .withColumn("date_of_birth", to_date(col("date_of_birth")))
    # Padronização de nomes para reconciliação entre fontes
    .withColumn("player_name_clean", lower(trim(col("name"))))
    # Completude: Imputação de valor seguro em nulos categóricos
    .fillna({"sub_position": "Unknown", "foot": "Unknown"})
    # Regras de Negócio: Exclui cadastros sem identificador ou nome
    .filter(col("player_id").isNotNull() & col("name").isNotNull())
    # Auditoria
    .withColumn("_transformed_at", current_timestamp())
    .select(
        col("player_id"),
        col("name").alias("player_name"),
        col("player_name_clean"),
        col("date_of_birth"),
        col("country_of_citizenship"),
        col("position"),
        col("sub_position"),
        col("foot"),
        col("height_in_cm"),
        col("current_club_id"),
        col("current_club_name"),
        col("market_value_in_eur"),
        col("highest_market_value_in_eur"),
        col("_transformed_at"),
    )
)

df_players_silver.write.format("delta").mode("overwrite").saveAsTable(
    "silver.players"
)
print(
    f"[OK] Tabela 'silver.players' criada com"
    f" {spark.table('silver.players').count():,} atletas."
)

In [0]:
print("Processando atuações por partida (Kaggle: appearances)...")

df_appearances_silver = (
    spark.table("bronze.appearances")
    # Unicidade
    .dropDuplicates(["appearance_id"])
    # Consistência: Conversão para data formal
    .withColumn("date", to_date(col("date"), "yyyy-MM-dd"))
    .withColumn("player_name_clean", lower(trim(col("player_name"))))
    # Validação de Outliers / Regras de Negócio: Descarte de minutos negativos
    .filter(col("minutes_played") >= 0)
    # Completude: Métricas de jogo não podem ser nulas (substituição por 0)
    .fillna(
        {"goals": 0, "assists": 0, "yellow_cards": 0, "red_cards": 0},
        subset=["goals", "assists", "yellow_cards", "red_cards"],
    )
    # Auditoria
    .withColumn("_transformed_at", current_timestamp())
    .select(
        col("appearance_id"),
        col("game_id"),
        col("player_id"),
        col("player_name"),
        col("player_name_clean"),
        col("date"),
        col("player_club_id"),
        col("competition_id"),
        col("goals"),
        col("assists"),
        col("minutes_played"),
        col("yellow_cards"),
        col("red_cards"),
        col("_transformed_at"),
    )
)

df_appearances_silver.write.format("delta").mode("overwrite").saveAsTable(
    "silver.appearances"
)
print(
    f"[OK] Tabela 'silver.appearances' criada com"
    f" {spark.table('silver.appearances').count():,} atuações."
)

In [0]:
print("Processando dados de equipes (Kaggle: clubs)...")

df_clubs_silver = (
    spark.table("bronze.clubs")
    # Unicidade
    .dropDuplicates(["club_id"])
    .filter(col("club_id").isNotNull() & col("name").isNotNull())
    .withColumn("club_name_clean", trim(col("name")))
    .withColumn("_transformed_at", current_timestamp())
    .select(
        col("club_id"),
        col("club_code"),
        col("name").alias("club_name"),
        col("domestic_competition_id"),
        col("total_market_value"),
        col("squad_size"),
        col("average_age"),
        col("stadium_name"),
        col("_transformed_at"),
    )
)

df_clubs_silver.write.format("delta").mode("overwrite").saveAsTable(
    "silver.clubs"
)
print(
    f"[OK] Tabela 'silver.clubs' criada com"
    f" {spark.table('silver.clubs').count():,} clubes."
)

In [0]:
from pyspark.sql.functions import col, current_timestamp, round, to_date, trim, year

print("Processando histórico de transferências (Kaggle: transfers)...")

df_transfers_silver = (
    spark.table("bronze.transfers")
    # 1. Unicidade: remove duplicatas por atleta, data e destino
    .dropDuplicates(["player_id", "transfer_date", "to_club_id"])
    # 2. Tipagem: conversão explícita de campos de data
    .withColumn("transfer_date", to_date(col("transfer_date"), "yyyy-MM-dd"))
    .withColumn("transfer_year", year(col("transfer_date")))
    # 3. Padronização e limpeza de textos
    .withColumn("player_name_clean", trim(col("player_name")))
    .withColumn("from_club_name", trim(col("from_club_name")))
    .withColumn("to_club_name", trim(col("to_club_name")))
    # 4. Tratamento de nulos em valores financeiros
    .fillna({"transfer_fee": 0, "market_value_in_eur": 0})
    # 5. Regras de consistência: atletas com chave válida e transferências datadas
    .filter(col("player_id").isNotNull() & col("transfer_date").isNotNull())
    # 6. Auditoria de linhagem
    .withColumn("_transformed_at", current_timestamp())
    .select(
        col("player_id"),
        col("player_name"),
        col("player_name_clean"),
        col("transfer_date"),
        col("transfer_year"),
        col("transfer_season"),
        col("from_club_id"),
        col("from_club_name"),
        col("to_club_id"),
        col("to_club_name"),
        col("transfer_fee"),
        col("market_value_in_eur"),
        col("_transformed_at"),
    )
)

# Persistência Delta Lake
df_transfers_silver.write.format("delta").mode("overwrite").saveAsTable(
    "silver.transfers"
)
print(
    f"[OK] Tabela 'silver.transfers' criada com"
    f" {spark.table('silver.transfers').count():,} movimentações de mercado."
)

In [0]:
print("--- RELATÓRIO DE AUDITORIA E QUALIDADE (CAMADA SILVER) ---")

# 1. Teste de Completude e Chaves Nulas
nulos_players = spark.table("silver.players").filter(col("player_id").isNull()).count()
nulos_fotmob = spark.table("silver.fotmob_ratings").filter(col("rating").isNull()).count()

# 2. Teste de Acurácia de Escala
inconsistencias_rating = (
    spark.table("silver.fotmob_ratings")
    .filter((col("rating") < 0) | (col("rating") > 10))
    .count()
)

# 3. Teste de Regras de Negócio
minutos_invalidos = (
    spark.table("silver.appearances")
    .filter(col("minutes_played") < 0)
    .count()
)

print(f"1. Completude: Jogadores sem ID: {nulos_players} | Ratings nulos: {nulos_fotmob}")
print(f"2. Acurácia: Notas fora do intervalo 0-10: {inconsistencias_rating}")
print(f"3. Consistência: Partidas com minutos negativos: {minutos_invalidos}")

assert (
    nulos_players == 0
    and nulos_fotmob == 0
    and inconsistencias_rating == 0
    and minutos_invalidos == 0
), "Falha detectada nos critérios de qualidade de dados!"

print("\n[SUCESSO] Todas as regras de qualidade foram satisfeitas integralmente.")